In [1]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape 
from pyvis.network import Network
import networkx as nx
from shapely.geometry import LineString
from shapely import wkt
from networkx.algorithms.simple_paths import shortest_simple_paths

In [2]:
with open('data/metro_stops.json') as f:
    metro_stops = json.load(f)

with open('data/metro_trajectories.json') as f:
    metro_trajectories = json.load(f)

excluded = ['FM','L10N','L10S','L11','L9N','L9S','L9NL10N','L9SL10S','TM']


In [3]:
df_stops = []
for stop in metro_stops['features']:
    properties = stop.get('properties', {})
    nom_estacio = properties.get('NOM_ESTACIO')
    df_stops.append({
            'name': nom_estacio,
            'geometry': shape(stop['geometry'])
        })

df_stops = pd.DataFrame(df_stops)
geo_df_stops = gpd.GeoDataFrame(df_stops)
geo_df_stops = geo_df_stops.set_crs(4326)
geo_df_stops

,name,geometry
0,Hospital de Bellvitge,POINT (2.10724 41.34468)
1,Bellvitge,POINT (2.11092 41.35097)
2,Av. Carrilet,POINT (2.10265 41.35852)
3,Rambla Just Oliveras,POINT (2.09975 41.36409)
4,Can Serra,POINT (2.10275 41.36769)
...,...,...
135,Torre Baró | Vallbona,POINT (2.17988 41.4592)
136,Ciutat Meridiana,POINT (2.17465 41.46081)
137,Can Cuiàs,POINT (2.17306 41.46241)
138,Telefèric Parc de Montjuïc,POINT (2.16327 41.36896)


In [4]:
df_trajectories_all = []
for trajectory in metro_trajectories['features']:
    properties = trajectory.get('properties', {})
    tram = properties.get('NOM_TRAM_LINIA')
    nom_linia = properties.get('NOM_LINIA')
    df_trajectories_all.append({
            'tram': tram,
            'nom_inici': properties.get('NOM_ESTACIO_INI'),
            'nom_fi': properties.get('NOM_ESTACIO_FI'),
            'linia': nom_linia,
            'geometry': shape(trajectory['geometry'])
        })
df_trajectories_all = pd.DataFrame(df_trajectories_all)
geo_df_trajectories_all = gpd.GeoDataFrame(df_trajectories_all, geometry='geometry')
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Inici') == False]
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Final') == False]
geo_df_trajectories_all = geo_df_trajectories_all[~geo_df_trajectories_all['linia'].isin(excluded)]
geo_df_trajectories_all = geo_df_trajectories_all.set_crs(4326)
geo_df_trajectories_all['distance'] = (geo_df_trajectories_all.to_crs(25831).geometry.length)
geo_df_trajectories_all

,tram,nom_inici,nom_fi,linia,geometry,distance
1,Hospital de Bellvitge - Bellvitge,Hospital de Bellvitge,Bellvitge,L1,"MULTILINESTRING ((2.10724 41.34468, 2.10773 41...",882.209743
2,Bellvitge - Av. Carrilet,Bellvitge,Av. Carrilet,L1,"MULTILINESTRING ((2.11092 41.35098, 2.11052 41...",1133.654863
3,Av. Carrilet - Rambla Just Oliveras,Av. Carrilet,Rambla Just Oliveras,L1,"MULTILINESTRING ((2.10263 41.35855, 2.10202 41...",660.855917
4,Rambla Just Oliveras - Can Serra,Rambla Just Oliveras,Can Serra,L1,"MULTILINESTRING ((2.09975 41.36409, 2.09946 41...",572.330154
5,Can Serra - Florida,Can Serra,Florida,L1,"MULTILINESTRING ((2.10276 41.36769, 2.10381 41...",616.671198
...,...,...,...,...,...,...
122,Virrei Amat - Vilapicina,Virrei Amat,Vilapicina,L5,"MULTILINESTRING ((2.17491 41.4297, 2.17486 41....",636.238350
123,Vilapicina - Horta,Vilapicina,Horta,L5,"MULTILINESTRING ((2.16762 41.43047, 2.16564 41...",683.078591
124,Horta - El Carmel,Horta,El Carmel,L5,"MULTILINESTRING ((2.15997 41.42969, 2.15974 41...",851.210213
125,El Carmel - El Coll | La Teixonera,El Carmel,El Coll | La Teixonera,L5,"MULTILINESTRING ((2.1551 41.42445, 2.15513 41....",838.886751


In [5]:
joined = geo_df_trajectories_all.merge(geo_df_stops, left_on='nom_inici', right_on='name', how='left')
joined.rename(columns={'geometry_x': 'trajectory_geometry', 'geometry_y': 'starting_geometry'}, inplace=True)
joined = joined.merge(geo_df_stops, left_on='nom_fi', right_on='name', how='left')
joined.rename(columns={'geometry': 'ending_geometry'}, inplace=True)
joined.drop(columns=['name_x', 'name_y',], inplace=True)
joined

,tram,nom_inici,nom_fi,linia,trajectory_geometry,distance,starting_geometry,ending_geometry
0,Hospital de Bellvitge - Bellvitge,Hospital de Bellvitge,Bellvitge,L1,"MULTILINESTRING ((2.10724 41.34468, 2.10773 41...",882.209743,POINT (2.10724 41.34468),POINT (2.11092 41.35097)
1,Bellvitge - Av. Carrilet,Bellvitge,Av. Carrilet,L1,"MULTILINESTRING ((2.11092 41.35098, 2.11052 41...",1133.654863,POINT (2.11092 41.35097),POINT (2.10265 41.35852)
2,Av. Carrilet - Rambla Just Oliveras,Av. Carrilet,Rambla Just Oliveras,L1,"MULTILINESTRING ((2.10263 41.35855, 2.10202 41...",660.855917,POINT (2.10265 41.35852),POINT (2.09975 41.36409)
3,Rambla Just Oliveras - Can Serra,Rambla Just Oliveras,Can Serra,L1,"MULTILINESTRING ((2.09975 41.36409, 2.09946 41...",572.330154,POINT (2.09975 41.36409),POINT (2.10275 41.36769)
4,Can Serra - Florida,Can Serra,Florida,L1,"MULTILINESTRING ((2.10276 41.36769, 2.10381 41...",616.671198,POINT (2.10275 41.36769),POINT (2.11003 41.36832)
...,...,...,...,...,...,...,...,...
113,Virrei Amat - Vilapicina,Virrei Amat,Vilapicina,L5,"MULTILINESTRING ((2.17491 41.4297, 2.17486 41....",636.238350,POINT (2.17491 41.4297),POINT (2.16762 41.43047)
114,Vilapicina - Horta,Vilapicina,Horta,L5,"MULTILINESTRING ((2.16762 41.43047, 2.16564 41...",683.078591,POINT (2.16762 41.43047),POINT (2.15997 41.42969)
115,Horta - El Carmel,Horta,El Carmel,L5,"MULTILINESTRING ((2.15997 41.42969, 2.15974 41...",851.210213,POINT (2.15997 41.42969),POINT (2.1551 41.42445)
116,El Carmel - El Coll | La Teixonera,El Carmel,El Coll | La Teixonera,L5,"MULTILINESTRING ((2.1551 41.42445, 2.15513 41....",838.886751,POINT (2.1551 41.42445),POINT (2.14835 41.422)


# Apply algorithm (real edges)

In [6]:
G = nx.Graph()
stations = set(joined['nom_inici']).union(set(joined['nom_fi']))

for station in stations:
    G.add_node(station,label=station,title=station)

for _, row in joined.iterrows():
    G.add_edge(row['nom_inici'],row['nom_fi'],weight=row['distance'])


G.add_node('Bus Entry BCN', label='Bus Entry BCN', title='Bus Entry BCN')


In [7]:
geo_df_trajectories_all

,tram,nom_inici,nom_fi,linia,geometry,distance
1,Hospital de Bellvitge - Bellvitge,Hospital de Bellvitge,Bellvitge,L1,"MULTILINESTRING ((2.10724 41.34468, 2.10773 41...",882.209743
2,Bellvitge - Av. Carrilet,Bellvitge,Av. Carrilet,L1,"MULTILINESTRING ((2.11092 41.35098, 2.11052 41...",1133.654863
3,Av. Carrilet - Rambla Just Oliveras,Av. Carrilet,Rambla Just Oliveras,L1,"MULTILINESTRING ((2.10263 41.35855, 2.10202 41...",660.855917
4,Rambla Just Oliveras - Can Serra,Rambla Just Oliveras,Can Serra,L1,"MULTILINESTRING ((2.09975 41.36409, 2.09946 41...",572.330154
5,Can Serra - Florida,Can Serra,Florida,L1,"MULTILINESTRING ((2.10276 41.36769, 2.10381 41...",616.671198
...,...,...,...,...,...,...
122,Virrei Amat - Vilapicina,Virrei Amat,Vilapicina,L5,"MULTILINESTRING ((2.17491 41.4297, 2.17486 41....",636.238350
123,Vilapicina - Horta,Vilapicina,Horta,L5,"MULTILINESTRING ((2.16762 41.43047, 2.16564 41...",683.078591
124,Horta - El Carmel,Horta,El Carmel,L5,"MULTILINESTRING ((2.15997 41.42969, 2.15974 41...",851.210213
125,El Carmel - El Coll | La Teixonera,El Carmel,El Coll | La Teixonera,L5,"MULTILINESTRING ((2.1551 41.42445, 2.15513 41....",838.886751


In [8]:
point_0 = shape({
    'type': 'Point',
    'coordinates':  (2.184459824628127, 41.401546206394364)
})

point1 = shape({
    'type': 'Point',
    'coordinates': (2.167342560951652, 41.38704037682107)
})

point2 = shape({
    'type': 'Point',
    'coordinates': (2.1768302000749906, 41.39610145325573)
})

# build points df
points = [point_0, point1, point2]
geo_df_points = gpd.GeoDataFrame(geometry=points, crs=4326)
geo_df_points['label'] = ['Point 0', 'Point 1', 'Point 2']

# buld bus trajectories df
geo_df_points_m = geo_df_points.to_crs(25831)
geo_df_points_m["x"] = geo_df_points_m.geometry.x
geo_df_points_m_sorted = geo_df_points_m.sort_values("x", ascending=False).reset_index(drop=True)
geo_df_points_m_sorted


rows = []

for i in range(len(geo_df_points_m_sorted) - 1):
    p1 = geo_df_points_m_sorted.iloc[i]
    p2 = geo_df_points_m_sorted.iloc[i + 1]

    line = LineString([p1.geometry, p2.geometry])
    distance = p1.geometry.distance(p2.geometry)

    rows.append({
        "from_label": p1["label"],
        "to_label": p2["label"],
        "distance": distance,
        "geometry": line
    })

geo_df_points_trajectory = gpd.GeoDataFrame(rows, crs=25831)
geo_df_points_trajectory


# project metro stops once (IMPORTANT: do this outside loop)
geo_df_stops_metric = geo_df_stops.to_crs(25831).copy()

snap_results = []
G_temp = G.copy()
print(geo_df_points)

for i, row in geo_df_points.iterrows():
    if i == 0:
        start_node = row["label"]
        G_temp.add_node(start_node, label=start_node, title=start_node)
        continue
    point_gdf = gpd.GeoSeries([row.geometry], crs=4326).to_crs(25831)
    point_metric = point_gdf.iloc[0]
    geo_df_stops_metric["distance_to_point"] = geo_df_stops_metric.geometry.distance(point_metric)

    closest_idx = geo_df_stops_metric["distance_to_point"].idxmin()
    closest_stop = geo_df_stops_metric.loc[closest_idx]
    snap_results.append({
        "point_index": i,
        "point_label": row["label"],
        "closest_stop": closest_stop["name"],
        "distance": closest_stop["distance_to_point"]
    })
    G_temp.add_node(row["label"], label=row["label"], title=row["label"])
    G_temp.add_edge(row["label"],closest_stop["name"],weight=closest_stop["distance_to_point"],title="Snap to metro")


for _, row in geo_df_points_trajectory.iterrows():
    G_temp.add_edge(row["from_label"], row["to_label"], weight=row["distance"], title="Bus trajectory")

                   geometry    label
0  POINT (2.18446 41.40155)  Point 0
1  POINT (2.16734 41.38704)  Point 1
2   POINT (2.17683 41.3961)  Point 2


In [9]:
geo_df_points_trajectory

,from_label,to_label,distance,geometry
0,Point 0,Point 2,878.763649,"LINESTRING (431829.475 4583654.925, 431185.971..."
1,Point 2,Point 1,1281.108052,"LINESTRING (431185.971 4583056.488, 430383.164..."


In [10]:
destination1 = shape({
    'type': 'Point',
    'coordinates': (2.1130282510923517, 41.38913433225749)
}) # fib

destination2 = shape({
    'type': 'Point',
    'coordinates': (2.1455911047955754, 41.43508184671605)
}) # mundet

destination3 = shape({
    'type': 'Point',
    'coordinates': (2.198402153798261, 41.40288229103046)
}) # poble nou

destination4 = shape({
    'type': 'Point',
    'coordinates':  (2.1943145021006196, 41.396975996699524)
}) # carrer pujadas 77

results = []

geo_df_metric = geo_df_stops.to_crs(25831)


destinations = [destination1, destination2, destination3, destination4]
for dest in destinations:
    destination_name = f"Destination {destinations.index(dest)+1}"
    G_exp = G_temp.copy()  # or rebuild from scratch if needed
    destination_gdf = gpd.GeoSeries([dest], crs=4326).to_crs(25831)
    geo_df_stops['distance_to_destination'] = geo_df_metric.geometry.distance(destination_gdf.iloc[0])
    geo_df_stops['closest_stop_end'] = (geo_df_stops['distance_to_destination']== geo_df_stops['distance_to_destination'].min())
    closest_stop_end = geo_df_stops[geo_df_stops['distance_to_destination'] == geo_df_stops['distance_to_destination'].min()]

    G_exp.add_node('End', label='End', title='Destination', size=200, color='blue')
    G_exp.add_edge(closest_stop_end.iloc[0]['name'], 'End', weight=closest_stop_end.iloc[0]['distance_to_destination'], color='black', title='Walk to Destination')
    
    paths = list(
    shortest_simple_paths(
        G_exp,
        source=start_node,
        target="End",
        weight="weight"
    )
    )

    top_3_paths = paths[:20]

    top_3_results = []

    i = 1
    for p in top_3_paths:

        total_dist = sum(
            G_exp[p[i]][p[i+1]]["weight"]
            for i in range(len(p) - 1)
        )

        bus_dist = sum(
            G_exp[p[i]][p[i+1]]["weight"]
            for i in range(len(p) - 1)
            if G_exp[p[i]][p[i+1]].get("title") == "Bus trajectory"
        )

        metro_and_walk_dist = total_dist - bus_dist

        results.append({
            'Destination': destination_name,
            'first_stop': start_node,
            'last_stop': closest_stop_end.iloc[0]['name'],
            'path_id':i,
            "path": p,
            "total_distance": total_dist,
            "bus_distance": bus_dist,
            "metro_and_walk_distance": metro_and_walk_dist
        })
        i += 1

    
    
    # path = nx.shortest_path(
    #     G_exp,
    #     source=start_node,
    #     target='End',
    #     weight='weight'
    # )
    # distance = nx.shortest_path_length(
    #     G_exp,
    #     source=start_node,
    #     target='End',
    #     weight='weight'
    # )
    # results.append({
    #     'first_stop': start_node,
    #     'last_stop': closest_stop_end.iloc[0]['name'],
    #     'path': path,
    #     'distance': distance
    # })

In [11]:
results_df = pd.DataFrame(results)
# delete start and end from path
results_df['path'] = results_df['path'].apply(lambda x: x[1:-1])
results_df['number_of_stops'] = results_df['path'].apply(len)
results_df

,Destination,first_stop,last_stop,path_id,path,total_distance,bus_distance,metro_and_walk_distance,number_of_stops
0,Destination 1,Point 0,Palau Reial,1,"[Point 2, Tetuan, Passeig de Gràcia, Diagonal,...",7848.293369,878.763649,6969.529720,11
1,Destination 1,Point 0,Palau Reial,2,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",8482.433548,878.763649,7603.669899,13
2,Destination 1,Point 0,Palau Reial,3,"[Point 2, Point 1, Catalunya, Universitat, Urg...",8724.628413,2159.871701,6564.756712,13
3,Destination 1,Point 0,Palau Reial,4,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",8909.711337,878.763649,8030.947687,14
4,Destination 1,Point 0,Palau Reial,5,"[Point 2, Tetuan, Passeig de Gràcia, Catalunya...",8997.701142,878.763649,8118.937492,14
...,...,...,...,...,...,...,...,...,...
75,Destination 4,Point 0,Bogatell,16,"[Point 2, Tetuan, Monumental, Sagrada Família,...",11252.975730,878.763649,10374.212080,16
76,Destination 4,Point 0,Bogatell,17,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",11731.316426,878.763649,10852.552777,17
77,Destination 4,Point 0,Bogatell,18,"[Point 2, Point 1, Catalunya, Liceu, Drassanes...",12251.326442,2159.871701,10091.454741,17
78,Destination 4,Point 0,Bogatell,19,"[Point 2, Tetuan, Passeig de Gràcia, Diagonal,...",12252.348524,878.763649,11373.584875,18


In [12]:
rows = []
for r in results:
    path = r["path"].copy()
    for i in range(len(path) - 1):
        rows.append({
            "first_stop": r["first_stop"],
            "last_stop": r["last_stop"],
            "path_id": r["path_id"],
            'order': i,
            "formatted path": f"{path[i]} - {path[i+1]}",
        })
        
df_paths = pd.DataFrame(rows)
df_paths["edge_key"] = df_paths["formatted path"].apply(
    lambda x: " - ".join(sorted(x.split(" - ")))
)

joined["edge_key"] = joined["tram"].apply(
    lambda x: " - ".join(sorted(x.split(" - ")))
)

df_paths_joined = df_paths.merge(
    joined[["edge_key", "tram", "linia"]],
    on="edge_key",
    how="left"
)

df_paths_joined = df_paths_joined.sort_values(
    ["first_stop", "last_stop", "path_id", "order"]
)

df_paths_joined["prev_line"] = (
    df_paths_joined.groupby("path_id")["linia"].shift(1)
)

df_paths_joined["exchange"] = (
    df_paths_joined["linia"] != df_paths_joined["prev_line"]
)

df_paths_joined.loc[
    df_paths_joined["prev_line"].isna(),
    "exchange"
] = False


exchange_counts = (
    df_paths_joined
    .groupby(["first_stop", "last_stop", "path_id"])["exchange"]
    .sum()
    .reset_index(name="num_exchanges")
)

exchange_counts['num_exchanges'] = exchange_counts['num_exchanges'] - 1

# number_lines = df_paths_joined.groupby(["first_stop", "last_stop",'path_id'])['linia'].nunique().reset_index()
# number_lines.rename(columns={'linia': 'number_of_lines'}, inplace=True)
# number_lines

In [13]:
results_df = results_df.merge(
    exchange_counts,
    on=["first_stop", "last_stop",'path_id'],
    how="left"
)
results_df

,Destination,first_stop,last_stop,path_id,path,total_distance,bus_distance,metro_and_walk_distance,number_of_stops,num_exchanges
0,Destination 1,Point 0,Palau Reial,1,"[Point 2, Tetuan, Passeig de Gràcia, Diagonal,...",7848.293369,878.763649,6969.529720,11,3
1,Destination 1,Point 0,Palau Reial,2,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",8482.433548,878.763649,7603.669899,13,2
2,Destination 1,Point 0,Palau Reial,3,"[Point 2, Point 1, Catalunya, Universitat, Urg...",8724.628413,2159.871701,6564.756712,13,1
3,Destination 1,Point 0,Palau Reial,4,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",8909.711337,878.763649,8030.947687,14,3
4,Destination 1,Point 0,Palau Reial,5,"[Point 2, Tetuan, Passeig de Gràcia, Catalunya...",8997.701142,878.763649,8118.937492,14,3
...,...,...,...,...,...,...,...,...,...,...
75,Destination 4,Point 0,Bogatell,16,"[Point 2, Tetuan, Monumental, Sagrada Família,...",11252.975730,878.763649,10374.212080,16,1
76,Destination 4,Point 0,Bogatell,17,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",11731.316426,878.763649,10852.552777,17,4
77,Destination 4,Point 0,Bogatell,18,"[Point 2, Point 1, Catalunya, Liceu, Drassanes...",12251.326442,2159.871701,10091.454741,17,3
78,Destination 4,Point 0,Bogatell,19,"[Point 2, Tetuan, Passeig de Gràcia, Diagonal,...",12252.348524,878.763649,11373.584875,18,5


['Point 2', 'Tetuan', 'Passeig de Gràcia', 'Urquinaona', 'Jaume I', 'Barceloneta', 'Ciutadella | Vila Olímpica', 'Bogatell']

In [14]:
geo_df_trajectories_all['distance'].mean()

np.float64(723.7829116056531)

In [15]:
results_df['number_of_bus_stops'] = results_df['path'].apply(lambda x: len([stop for stop in x if stop in geo_df_points['label'].values]))
results_df['total_objective'] = results_df['metro_and_walk_distance'] + (results_df['number_of_stops'] * geo_df_trajectories_all['distance'].mean() )+ (500 * results_df['num_exchanges'])
results_df

,Destination,first_stop,last_stop,path_id,path,total_distance,bus_distance,metro_and_walk_distance,number_of_stops,num_exchanges,number_of_bus_stops,total_objective
0,Destination 1,Point 0,Palau Reial,1,"[Point 2, Tetuan, Passeig de Gràcia, Diagonal,...",7848.293369,878.763649,6969.529720,11,3,1,16431.141747
1,Destination 1,Point 0,Palau Reial,2,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",8482.433548,878.763649,7603.669899,13,2,1,18012.847750
2,Destination 1,Point 0,Palau Reial,3,"[Point 2, Point 1, Catalunya, Universitat, Urg...",8724.628413,2159.871701,6564.756712,13,1,2,16473.934563
3,Destination 1,Point 0,Palau Reial,4,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",8909.711337,878.763649,8030.947687,14,3,1,19663.908450
4,Destination 1,Point 0,Palau Reial,5,"[Point 2, Tetuan, Passeig de Gràcia, Catalunya...",8997.701142,878.763649,8118.937492,14,3,1,19751.898255
...,...,...,...,...,...,...,...,...,...,...,...,...
75,Destination 4,Point 0,Bogatell,16,"[Point 2, Tetuan, Monumental, Sagrada Família,...",11252.975730,878.763649,10374.212080,16,1,1,22454.738666
76,Destination 4,Point 0,Bogatell,17,"[Point 2, Tetuan, Passeig de Gràcia, Universit...",11731.316426,878.763649,10852.552777,17,4,1,25156.862274
77,Destination 4,Point 0,Bogatell,18,"[Point 2, Point 1, Catalunya, Liceu, Drassanes...",12251.326442,2159.871701,10091.454741,17,3,2,23895.764238
78,Destination 4,Point 0,Bogatell,19,"[Point 2, Tetuan, Passeig de Gràcia, Diagonal,...",12252.348524,878.763649,11373.584875,18,5,1,26901.677283


demanar top 5 shortest paths, fer que conti transbords, no linies uniques

# test proposed solution

In [28]:
import altair as alt

In [30]:
test = df_paths_joined[df_paths_joined['destination_name'] == 'Destination 4']
test = df_paths_joined[df_paths_joined['path_id'] == 1]
test = joined[joined['edge_key'].isin(test['edge_key'])]
neigh = pd.read_csv("data/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)



KeyError: 'destination_name'

In [ ]:
test.rename(columns={'trajectory_geometry': 'geometry'},inplace=True)

In [ ]:
test = test[['linia', 'geometry']]
test

In [ ]:
neigh_layer = alt.Chart(neigh).mark_geoshape(
    stroke="black",
    fill="lightblue"
).properties(
    width=600,
    height=600)


trajectory_layer = alt.Chart(test).mark_geoshape(
    strokeWidth=3,filled = False).encode(
        color=alt.Color('linia:N',scale = alt.Scale(domain=['L1', 'L2', 'L3','L5'], range=['red', 'purple', 'green','blue']))).properties(
    width=600,
    height=600
)

point_layer = alt.Chart(geo_df_stops).mark_geoshape(
    stroke="black").encode(tooltip=['name'])



(neigh_layer + trajectory_layer + point_layer).resolve_scale(color='independent')

In [ ]:
['Point 2', 'Tetuan', 'Passeig de Gràcia', 'Catalunya', 'Universitat', 'Urgell', 'Rocafort', 'Espanya', 'Tarragona', 'Sants Estació', 'Plaça del Centre', 'Les Corts', 'Maria Cristina', 'Palau Reial']

In [ ]:
bus_objective = sum(geo_df_points_trajectory['distance']) + 100 *(len(geo_df_points_trajectory) -1)
bus_objective

objective function (metro) = sum avg(dist) * nstops + dist total + sum(numberof_lines - 1 * 500)

objective function (bus) total dist + num stops * 100